## Setup

Before you run the first code cell:

- **API key:** This notebook sends requests to a model through OpenRouter. Store your key in a `.env` file as `OPENROUTER_API_KEY=...`. If the key is missing, the cell asks you to paste it, and your input is hidden.
- **Model:** The `MODEL` value is also read from `.env`. If it isn't set, the notebook uses a default model.
- **Keep your key private.** Don't share it or commit it to a shared repository.


In [1]:
import os
import getpass
from datetime import datetime
from dotenv import load_dotenv

load_dotenv(override=True)
if not os.getenv("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass.getpass("Enter your OpenRouter API key: ")

MODEL = os.getenv("MODEL", "openai/gpt-6-luna")

# Partial Prompt Templates in LangChain

**Partial prompt templates** let you fill in some placeholders early and leave the rest for later.

- 🔧 **Flexibility**: Set some template values early for a head start.

- **Two Kinds of Partial Value**:
   1. 🧵 **Strings**: Lock in known values upfront.
   2. 🔄 **Functions**: Use for values that change, like the weather.

- **Three Ways to Set Them** (all shown below):
   1. `.partial()` on an existing template.
   2. `partial_variables=` when you create a template.
   3. `PromptTemplate.from_template()`, a shortcut for creating a template from a string.

- 🌍 **Real-World Use**: Pre-fill static info like a name and an activity, and add dynamic details like the time later.

Partial templates keep the prompt's structure while streamlining the prompting process.


> **Note:** You may see a `UserWarning` about Pydantic V1 when running on Python 3.14 or later. It comes from LangChain's internal compatibility code. It does not change the results in this notebook.


In [2]:
from langchain_core.prompts import PromptTemplate

/Users/moche/IADT_Local/26_27/CC-AIEng/.venv/lib/python3.14/site-packages/langchain_core/utils/pydantic.py:42: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1 import BaseModel as BaseModelV1


## What Is a `PromptTemplate`?

A **prompt template** is a prompt with blanks in it. The blanks are written as names in curly braces, such as `{UserName}`. When you fill them in, you get a finished prompt.

In the next cell, `{UserName}` and `{UserActivity}` are the **input variables**: the values you must supply before the prompt is complete. `input_variables` lists them. This argument is optional, because LangChain can work them out from the template, but listing them makes the intent clear.


In [3]:
prompt = PromptTemplate(
    template="Hello, {UserName}! Hope you're enjoying {UserActivity}.",
    # input_variables is optional (LangChain can infer it). It is listed here to be explicit.
    input_variables=["UserName", "UserActivity"]
)

## Missing Values Raise an Error

`format()` needs a value for every input variable. The next cell leaves out `UserActivity`, so Python raises a `KeyError`. The `try`/`except` block catches the error and prints a readable message instead of crashing.


In [4]:
try:
    prompt.format(UserName="Alice")
except KeyError as e:
    print(f"KeyError: {e} is missing. format() needs a value for every input variable.")


KeyError: 'UserActivity' is missing. format() needs a value for every input variable.


## Filling in Some Values Early with `.partial()`

`.partial()` fixes some values now and leaves the rest for later. Here it fixes `UserName`.

The original `prompt` is **unchanged**. `.partial()` returns a new template, so `prompt` still needs both values.

The output of the next cell shows the result:
- `partial_variables` holds the value that is already filled in.
- `input_variables` lists what is still missing: `UserActivity`.


In [5]:
partial_prompt = prompt.partial(UserName="Alice")

In [6]:
partial_prompt

PromptTemplate(input_variables=['UserActivity'], input_types={}, partial_variables={'UserName': 'Alice'}, template="Hello, {UserName}! Hope you're enjoying {UserActivity}.")

In [7]:
print(partial_prompt.format(UserActivity="your time at the Wonderland Resort and chasing our rabbits"))

Hello, Alice! Hope you're enjoying your time at the Wonderland Resort and chasing our rabbits.


# Partial with Functions

Sometimes a value should be worked out when the prompt is used, not when the template is created. For that, pass a **function** instead of a string.

- 💡 **Evaluated at format time**: The function runs each time the prompt is formatted, so each prompt gets the current value.
- 🔁 **Pass the function, not its result**: Write `_get_weather`, not `_get_weather()`. The brackets would run the function once, when the template is created, and freeze its answer.
- ⚠️ **Limitations**: A string value stays fixed. A function's value can change from one call to the next, which is useful but can surprise you. A slow or failing function affects every prompt that uses it.

🌤️ -> 📜 Use a function for live data, such as the weather, so prompts reflect the current value.


> **Note:** `_get_weather` is a mock, not a real weather service. It picks one of four conditions based on the day of the month (`day % 4`), so the same day in another month gives the same weather, and the pattern repeats every four days. The outputs below show "rainy", which the mock returns on days 1, 5, 9, 13, 17, 21, 25 and 29. In a real application, this function would call a weather API.


## Two Ways to Add a Function

The next cell defines `_get_weather` and uses it in two equivalent ways:

1. **`.partial(weather=_get_weather)`** adds the function to an existing template.
2. **`partial_variables={"weather": _get_weather}`** sets it when the template is created.

Both fill in the weather in the same way, so the choice comes down to style. Use whichever reads better in your code. Neither one calls `_get_weather()`, because the brackets would freeze the value at creation.


In [8]:
def _get_weather():
    # For the sake of this example, let's assume this function fetches the current weather.
    # In a real-world scenario, this could be integrated with a weather API.
    weather_conditions = ["sunny", "rainy", "cloudy", "snowy"]

    return weather_conditions[datetime.now().day % 4]  # Just a mock way to change weather daily

prompt = PromptTemplate(
    template="On this {weather} day, tell me a story that is {mood}.",
    input_variables=["mood", "weather"]
)

partial_prompt = prompt.partial(weather=_get_weather)

print(partial_prompt.format(mood="heartwarming"))

prompt = PromptTemplate(
    template="On this {weather} day, tell me a story that is {mood}.",
    input_variables=["mood", "weather"],
    partial_variables={"weather": _get_weather}
)
print(prompt.format(mood="adventurous"))

On this rainy day, tell me a story that is heartwarming.
On this rainy day, tell me a story that is adventurous.


## A Shortcut: `from_template()`

`PromptTemplate.from_template()` creates a template from a string. It is shorter than `PromptTemplate(template=..., input_variables=...)`, because LangChain works out the input variables from the `{placeholders}`. It also accepts `partial_variables=`, as shown in the next cell.

The first assignment in that cell is overwritten by the second, so it has no effect on the output.


In [9]:
prompt = PromptTemplate.from_template(template="On this {weather} day, tell me a story that is {mood}.")

prompt = PromptTemplate.from_template(
    template="On this {weather} day, tell me a story that is {mood}.",
    partial_variables={"weather": _get_weather}
)

print(prompt.format(mood="adventurous"))

On this rainy day, tell me a story that is adventurous.


## Partial Prompts in an LCEL Chain

A **chain** links steps together, so the output of one step becomes the input of the next. In LangChain this style is called **LCEL** (LangChain Expression Language). The `|` symbol, called a *pipe*, joins the steps:

`story_prompt | llm | StrOutputParser()`

1. `story_prompt` fills in the partial `weather` value and the `mood` you provide.
2. `llm` sends the finished prompt to a model. This notebook uses `ChatOpenRouter`, so the call goes to an external service over the internet and uses your API key.
3. `StrOutputParser()` takes the model's reply and returns it as plain text.

So `chain.invoke` only needs `mood`. The weather is filled in before the model sees the prompt. The model is chosen with `MODEL` in the setup cell.


In [10]:
from langchain_core.output_parsers import StrOutputParser
from langchain_openrouter import ChatOpenRouter

llm = ChatOpenRouter(model=MODEL, temperature=0)

story_prompt = PromptTemplate(
    template="On this {weather} day, tell me a story that is {mood}.",
    input_variables=["mood", "weather"],
    partial_variables={"weather": _get_weather}
)

chain = story_prompt | llm | StrOutputParser()
print(chain.invoke({"mood": "heartwarming"}))

### The Umbrella Shop

On the rainiest street in town stood a little shop that sold umbrellas.

Its owner, Mr. Bell, had every kind: red ones, blue ones, striped ones, and one enormous yellow umbrella that could shelter three people and a very determined dog.

One wet afternoon, a girl named Mina rushed into the shop, dripping from her hair to her shoes.

“I’m sorry,” she said. “I don’t have any money. I just needed somewhere dry to wait.”

Mr. Bell looked up from mending a broken umbrella. “Then you’ve come to the right place.”

He poured her a cup of warm cocoa and sat her beside the window. While Mina warmed her hands, they watched the rain blur the streetlights.

After a while, Mina noticed the umbrellas leaning by the door. “What happens to the ones people leave behind?”

“Sometimes,” said Mr. Bell, “they find someone who needs them.”

Just then, the door swung open. An old man stood there, shivering beneath a newspaper. Mina looked at the enormous yellow umbrella.

“Could he borr

> **Note:** The story above was written by the model. The weather phrase was filled in by the prompt, not by the model. Two things to keep in mind:
>
> - **Variation:** The text can differ between runs. A low `temperature` reduces randomness, but it does not guarantee identical text.
> - **Critical thinking:** A prompt can contain any data you put into it. If `_get_weather` were replaced by a function that read private or sensitive information (for example, a student's personal details), that data would be sent to an external service. Check what leaves your machine before you send it.
